# 📱 대학생 스마트폰 사용 패턴과 우울증(PHQ-9) 관계 분석
**데이터셋**: StudentLife (Dartmouth University, 2013)  
**분석 내용**: 전처리 → 기술통계 → 시각화  

---
> ⚠️ **실행 순서**: 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)

## STEP 1. 데이터 다운로드 및 압축 해제
> 다트머스 공식 서버에서 직접 다운로드합니다. (약 600MB~5GB, 시간 소요 있음)

In [ ]:
import os

# 다운로드 (공식 다트머스 링크)
!wget -q --show-progress -O dataset.tar.bz2 \
    https://studentlife.cs.dartmouth.edu/dataset/dataset.tar.bz2

print('✅ 다운로드 완료')

In [ ]:
# 압축 해제
!tar -xjf dataset.tar.bz2
print('✅ 압축 해제 완료')

# 폴더 구조 확인
!find dataset -maxdepth 3 -type d | sort

## STEP 2. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (코랩 환경)
!apt-get -qq install fonts-nanum > /dev/null
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('✅ 라이브러리 로드 완료')

## STEP 3. 핵심 데이터 불러오기

| 파일 | 설명 | 변수 역할 |
|------|------|----------|
| `survey/PHQ-9.csv` | 우울증 척도 | **종속변수** |
| `sensing/phonelock/` | 화면 잠금/해제 횟수 → 스마트폰 사용 빈도 | **독립변수** |
| `sensing/phonecharge/` | 충전 패턴 → 사용 시간대 추정 | 보조 독립변수 |

In [ ]:
# ── PHQ-9 불러오기 ──────────────────────────────────
phq_path = 'dataset/survey/PHQ-9.csv'
phq = pd.read_csv(phq_path)
print('PHQ-9 컬럼:', phq.columns.tolist())
print(f'참가자 수: {phq.shape[0]}명')
phq.head()

In [ ]:
# ── 화면 잠금 데이터 불러오기 (전체 참가자 합치기) ──
import glob

lock_files = glob.glob('dataset/sensing/phonelock/phonelock_u*.csv')
lock_list = []

for f in lock_files:
    uid = int(os.path.basename(f).replace('phonelock_u', '').replace('.csv', ''))
    try:
        df = pd.read_csv(f, header=None, names=['time', 'lock_status'])
        df['uid'] = uid
        lock_list.append(df)
    except Exception as e:
        print(f'  ⚠️ uid {uid} 건너뜀: {e}')

lock_raw = pd.concat(lock_list, ignore_index=True)
print(f'✅ 화면 잠금 데이터: {lock_raw.shape[0]:,}행 / 참가자 {lock_raw.uid.nunique()}명')
lock_raw.head()

## STEP 4. 전처리
### 4-1. PHQ-9 전처리

In [ ]:
print('=== PHQ-9 원본 컬럼 ===', phq.columns.tolist())
print('\n=== 결측치 확인 ===')
print(phq.isnull().sum())
print('\n=== 데이터 타입 ===')
print(phq.dtypes)

In [ ]:
# PHQ-9 전처리
phq_clean = phq.copy()

# 컬럼명 소문자 통일
phq_clean.columns = [c.lower().strip() for c in phq_clean.columns]

# uid 컬럼 찾기 (uid, student_id 등 다양한 표기 대응)
uid_col = [c for c in phq_clean.columns if 'uid' in c or 'student' in c or 'id' in c]
print('ID 관련 컬럼:', uid_col)

# PHQ-9 총점 컬럼 찾기 또는 계산
score_cols = [c for c in phq_clean.columns if 'phq' in c or 'score' in c or 'total' in c]
print('점수 관련 컬럼:', score_cols)

# 숫자형 컬럼이 9개(문항)라면 합계 계산
numeric_cols = phq_clean.select_dtypes(include='number').columns.tolist()
print('숫자형 컬럼:', numeric_cols)

In [ ]:
# PHQ-9 총점 계산 (문항 9개 합산)
# StudentLife 데이터는 pre/post 두 시점이 있음
# 'uid' 컬럼 외 나머지 숫자 컬럼들이 문항 응답값

# uid 컬럼 설정
if uid_col:
    phq_clean = phq_clean.rename(columns={uid_col[0]: 'uid'})

# PHQ-9 문항 컬럼 찾기 (uid 제외 숫자형)
item_cols = [c for c in numeric_cols if c != 'uid' and c not in ['uid']]
print('문항 컬럼 후보:', item_cols)

# 총점 계산
if len(item_cols) > 0:
    phq_clean['phq9_total'] = phq_clean[item_cols].sum(axis=1)
    
    # 결측치 제거
    before = len(phq_clean)
    phq_clean = phq_clean.dropna(subset=['phq9_total'])
    print(f'결측치 제거: {before - len(phq_clean)}행 제거')
    
    # 이상치 확인 (PHQ-9 범위: 0~27)
    out_of_range = phq_clean[(phq_clean['phq9_total'] < 0) | (phq_clean['phq9_total'] > 27)]
    print(f'범위 이상치(0~27 벗어남): {len(out_of_range)}건')
    phq_clean = phq_clean[(phq_clean['phq9_total'] >= 0) & (phq_clean['phq9_total'] <= 27)]

print(f'\n✅ PHQ-9 전처리 완료: {len(phq_clean)}명')
phq_clean[['uid', 'phq9_total']].head(10)

### 4-2. 스마트폰 사용 패턴 전처리 (화면 잠금 → 일별 사용 횟수)

In [ ]:
# timestamp → datetime 변환
lock_raw['datetime'] = pd.to_datetime(lock_raw['time'], unit='s', errors='coerce')
lock_raw = lock_raw.dropna(subset=['datetime'])

# 날짜 추출
lock_raw['date'] = lock_raw['datetime'].dt.date

# unlock(잠금 해제) 이벤트만 = 실제 사용 시작 횟수
# lock_status: 0=잠금, 1=해제 (데이터에 따라 다를 수 있음)
print('lock_status 고유값:', lock_raw['lock_status'].unique())

# 해제 이벤트 필터링 (값이 1이거나 'unlock'인 경우)
unlock_events = lock_raw[lock_raw['lock_status'].astype(str).isin(['1', 'unlock', 'UNLOCK'])]
print(f'unlock 이벤트 수: {len(unlock_events):,}건')

In [ ]:
# 참가자별 일평균 잠금해제 횟수 계산
daily_unlock = unlock_events.groupby(['uid', 'date']).size().reset_index(name='unlock_count')

# 참가자별 평균
avg_unlock = daily_unlock.groupby('uid')['unlock_count'].agg(
    mean_unlock='mean',
    std_unlock='std',
    total_days='count'
).reset_index()

# 이상치 처리: IQR 방법
Q1 = avg_unlock['mean_unlock'].quantile(0.25)
Q3 = avg_unlock['mean_unlock'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = avg_unlock[(avg_unlock['mean_unlock'] < lower) | (avg_unlock['mean_unlock'] > upper)]
print(f'이상치 참가자: {len(outliers)}명 (IQR 기준, 제거하지 않고 표시만)')
print(f'정상 범위: {lower:.1f} ~ {upper:.1f}회/일')

avg_unlock['is_outlier'] = (avg_unlock['mean_unlock'] < lower) | (avg_unlock['mean_unlock'] > upper)

print(f'\n✅ 사용 패턴 집계 완료: {len(avg_unlock)}명')
avg_unlock.head(10)

### 4-3. PHQ-9 + 스마트폰 사용 패턴 병합

In [ ]:
# uid 기준 병합
merged = pd.merge(
    phq_clean[['uid', 'phq9_total']],
    avg_unlock[['uid', 'mean_unlock', 'std_unlock', 'total_days', 'is_outlier']],
    on='uid',
    how='inner'
)

print(f'병합 결과: {len(merged)}명 (PHQ-9 + 사용 패턴 모두 있는 참가자)')
print('\n결측치 확인:')
print(merged.isnull().sum())

# PHQ-9 심각도 분류
def phq_severity(score):
    if score <= 4:   return '없음 (0-4)'
    elif score <= 9: return '경미 (5-9)'
    elif score <= 14: return '중등도 (10-14)'
    elif score <= 19: return '중증 (15-19)'
    else:            return '최중증 (20-27)'

merged['phq_severity'] = merged['phq9_total'].apply(phq_severity)

print('\n분석용 최종 데이터:')
merged.head()

## STEP 5. 기술통계

In [ ]:
print('=' * 50)
print('📊 기술통계 요약')
print('=' * 50)

stats = merged[['phq9_total', 'mean_unlock']].describe().round(2)
stats.index = ['관측수', '평균', '표준편차', '최솟값', '1사분위', '중앙값', '3사분위', '최댓값']
stats.columns = ['PHQ-9 총점', '일평균 잠금해제 횟수']
print(stats.to_string())

print('\n📋 PHQ-9 심각도 분포:')
severity_count = merged['phq_severity'].value_counts()
severity_pct = (severity_count / len(merged) * 100).round(1)
for s, n in severity_count.items():
    print(f'  {s}: {n}명 ({severity_pct[s]}%)')

## STEP 6. 시각화

In [ ]:
# ── 그래프 1: PHQ-9 점수 분포 ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PHQ-9 우울증 점수 분포', fontsize=15, fontweight='bold')

# 히스토그램
axes[0].hist(merged['phq9_total'], bins=15, color='#4A90D9', edgecolor='white', linewidth=0.8)
axes[0].axvline(merged['phq9_total'].mean(), color='red', linestyle='--',
                label=f'평균: {merged["phq9_total"].mean():.1f}')
axes[0].axvline(merged['phq9_total'].median(), color='orange', linestyle='--',
                label=f'중앙값: {merged["phq9_total"].median():.1f}')
axes[0].set_xlabel('PHQ-9 총점', fontsize=12)
axes[0].set_ylabel('참가자 수', fontsize=12)
axes[0].legend()
axes[0].set_title('점수 분포 (히스토그램)')

# 심각도 막대그래프
severity_order = ['없음 (0-4)', '경미 (5-9)', '중등도 (10-14)', '중증 (15-19)', '최중증 (20-27)']
colors = ['#2ECC71', '#F1C40F', '#E67E22', '#E74C3C', '#8E44AD']
counts = [merged['phq_severity'].value_counts().get(s, 0) for s in severity_order]
bars = axes[1].bar(range(len(severity_order)), counts, color=colors, edgecolor='white')
axes[1].set_xticks(range(len(severity_order)))
axes[1].set_xticklabels(severity_order, rotation=25, ha='right', fontsize=10)
axes[1].set_ylabel('참가자 수', fontsize=12)
axes[1].set_title('심각도별 분포')
for bar, count in zip(bars, counts):
    if count > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(count), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('plot1_phq_distribution.png', bbox_inches='tight')
plt.show()
print('✅ 그래프 1 저장 완료')

In [ ]:
# ── 그래프 2: 스마트폰 사용 빈도 분포 ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('스마트폰 사용 패턴 분포 (일평균 잠금해제 횟수)', fontsize=15, fontweight='bold')

# 히스토그램
normal = merged[~merged['is_outlier']]
outlier = merged[merged['is_outlier']]

axes[0].hist(normal['mean_unlock'], bins=12, color='#5DADE2', edgecolor='white', label='정상')
if len(outlier) > 0:
    axes[0].hist(outlier['mean_unlock'], bins=5, color='#E74C3C',
                edgecolor='white', alpha=0.7, label=f'이상치 ({len(outlier)}명)')
axes[0].set_xlabel('일평균 잠금해제 횟수', fontsize=12)
axes[0].set_ylabel('참가자 수', fontsize=12)
axes[0].legend()
axes[0].set_title('사용 빈도 분포')

# 박스플롯
axes[1].boxplot(merged['mean_unlock'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='#5DADE2', alpha=0.7),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', color='#E74C3C', markersize=6))
axes[1].set_ylabel('일평균 잠금해제 횟수', fontsize=12)
axes[1].set_title('박스플롯 (빨간 점 = 이상치)')
axes[1].set_xticklabels(['전체 참가자'])

plt.tight_layout()
plt.savefig('plot2_unlock_distribution.png', bbox_inches='tight')
plt.show()
print('✅ 그래프 2 저장 완료')

In [ ]:
# ── 그래프 3: 스마트폰 사용 vs PHQ-9 산점도 ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('스마트폰 사용 패턴 vs 우울증(PHQ-9) 관계', fontsize=15, fontweight='bold')

# 산점도 + 회귀선
from scipy import stats as sp_stats

x = merged['mean_unlock']
y = merged['phq9_total']
slope, intercept, r_value, p_value, std_err = sp_stats.linregress(x, y)

# 심각도별 색상 매핑
color_map = {'없음 (0-4)': '#2ECC71', '경미 (5-9)': '#F1C40F',
             '중등도 (10-14)': '#E67E22', '중증 (15-19)': '#E74C3C',
             '최중증 (20-27)': '#8E44AD'}
colors_scatter = merged['phq_severity'].map(color_map)

axes[0].scatter(x, y, c=colors_scatter, s=60, alpha=0.75, edgecolors='white', linewidth=0.5)
x_line = np.linspace(x.min(), x.max(), 100)
axes[0].plot(x_line, slope * x_line + intercept, color='navy', linewidth=2,
            label=f'회귀선 (r={r_value:.3f}, p={p_value:.3f})')
axes[0].set_xlabel('일평균 잠금해제 횟수', fontsize=12)
axes[0].set_ylabel('PHQ-9 총점', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_title('산점도 + 선형 회귀선')

# 레전드
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in color_map.items()
                  if k in merged['phq_severity'].values]
axes[0].legend(handles=legend_elements + [plt.Line2D([0],[0], color='navy',
              label=f'r={r_value:.3f}, p={p_value:.3f}')], fontsize=9)

# 심각도별 박스플롯
severity_order_present = [s for s in severity_order if s in merged['phq_severity'].values]
data_by_severity = [merged[merged['phq_severity'] == s]['mean_unlock'].values
                   for s in severity_order_present]
bp = axes[1].boxplot(data_by_severity, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
for patch, s in zip(bp['boxes'], severity_order_present):
    patch.set_facecolor(color_map[s])
    patch.set_alpha(0.75)
axes[1].set_xticklabels(severity_order_present, rotation=25, ha='right', fontsize=9)
axes[1].set_ylabel('일평균 잠금해제 횟수', fontsize=12)
axes[1].set_title('우울증 심각도별 스마트폰 사용 빈도')

plt.tight_layout()
plt.savefig('plot3_scatter_analysis.png', bbox_inches='tight')
plt.show()
print('✅ 그래프 3 저장 완료')
print(f'\n📌 상관계수 r = {r_value:.4f},  p값 = {p_value:.4f}')
if p_value < 0.05:
    print('  → p < 0.05: 통계적으로 유의미한 관계')
else:
    print('  → p ≥ 0.05: 통계적으로 유의미하지 않음 (표본 수 부족 가능)')

In [ ]:
# ── 그래프 4: 히트맵 (상관관계 요약) ─────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
fig.suptitle('변수 간 상관관계 히트맵', fontsize=14, fontweight='bold')

corr_cols = ['phq9_total', 'mean_unlock', 'std_unlock', 'total_days']
corr_labels = ['PHQ-9 총점', '평균 사용 횟수', '사용 변동성', '관측 일수']
corr_df = merged[corr_cols].rename(columns=dict(zip(corr_cols, corr_labels)))

corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
           center=0, vmin=-1, vmax=1, ax=ax,
           linewidths=0.5, square=True, mask=mask,
           annot_kws={'size': 11})
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plot4_heatmap.png', bbox_inches='tight')
plt.show()
print('✅ 그래프 4 저장 완료')

## STEP 7. 결과 저장

In [ ]:
# 분석용 최종 데이터 CSV 저장
merged.to_csv('studentlife_분석결과.csv', index=False, encoding='utf-8-sig')
print('✅ 분석결과 CSV 저장 완료: studentlife_분석결과.csv')

# 기술통계 저장
stats_summary = merged[['phq9_total', 'mean_unlock', 'std_unlock']].describe().round(3)
stats_summary.to_csv('기술통계_요약.csv', encoding='utf-8-sig')
print('✅ 기술통계 CSV 저장 완료: 기술통계_요약.csv')

print('\n📁 저장된 파일 목록:')
for f in ['studentlife_분석결과.csv', '기술통계_요약.csv',
          'plot1_phq_distribution.png', 'plot2_unlock_distribution.png',
          'plot3_scatter_analysis.png', 'plot4_heatmap.png']:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f'  ✔ {f} ({size:,} bytes)')

---
## ✅ 분석 완료

### 생성된 결과물
| 파일 | 내용 |
|------|------|
| `plot1_phq_distribution.png` | PHQ-9 점수 분포 + 심각도 막대그래프 |
| `plot2_unlock_distribution.png` | 스마트폰 사용 빈도 분포 + 이상치 시각화 |
| `plot3_scatter_analysis.png` | 산점도 + 회귀선 + 심각도별 박스플롯 |
| `plot4_heatmap.png` | 변수 간 상관관계 히트맵 |
| `studentlife_분석결과.csv` | 전처리된 최종 분석 데이터 |
| `기술통계_요약.csv` | 기술통계 요약표 |

### 다음 단계 옵션
- **회귀분석 추가**: `statsmodels`로 선형 회귀 상세 결과 (β, p값, R²)
- **충전 패턴 추가**: `sensing/phonecharge/` 데이터 병합
- **시계열 분석**: 10주 학기 동안 PHQ-9 변화 추이